# Chapter 2 — Forecast KPIs
## Case Study: Supply Chain KPI Audit — Which Metric to Trust?

**Reference:** Vandeput (2021), Chapter 2  
**Author:** Hanzila Bin Younus — EM CDE, University of Salzburg

---

### Industry Scenario

> **Company:** European logistics company managing 3 product categories:  
> fast-moving consumer goods, slow-moving industrial parts, and seasonal apparel.  
> The S&OP team is debating: *should we use MAE, MAPE, or RMSE to measure forecast*  
> *accuracy across our portfolio?* Each KPI gives a different answer — and each has  
> different consequences for safety stock and ordering decisions.

### Key Insight from Vandeput
> *"There is no one-size-fits-all indicator. Only experimentation will show you which*  
> *Key Performance Indicator is best for you."*

### KPIs Covered
| KPI | Formula | Sensitive to |
|-----|---------|-------------|
| Bias | $\frac{1}{n}\sum e_t$ | Direction of errors |
| MAE | $\frac{1}{n}\sum |e_t|$ | Average magnitude |
| RMSE | $\sqrt{\frac{1}{n}\sum e_t^2}$ | Large errors (outliers) |
| MAPE | $\frac{1}{n}\sum \frac{|e_t|}{d_t}$ | Scale — use with caution |
| FVA | $\frac{MAE_{Naïve} - MAE_{model}}{MAE_{Naïve}}$ | Value vs doing nothing |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fb',
                     'axes.spines.top': False, 'axes.spines.right': False})
C = {'demand': '#1a2332', 'forecast': '#0077b6', 'error': '#c0392b',
     'bias_pos': '#e07b00', 'bias_neg': '#6246ea'}

In [ ]:
# ── 1. THREE PRODUCT SCENARIOS ────────────────────────────────────────
np.random.seed(42)
n = 52

# Product A — fast-moving, stable, low noise
demand_A  = np.clip(1000 + np.random.normal(0, 60, n), 0, None)
forecast_A = demand_A * 0.97 + np.random.normal(0, 30, n)  # slight under-forecast

# Product B — slow-moving, occasional large spikes
demand_B  = np.clip(200 + np.random.normal(0, 20, n), 0, None)
demand_B[15] = 850; demand_B[38] = 920   # demand spikes
forecast_B = np.full(n, 220.0)  # flat forecast

# Product C — seasonal apparel (biased forecast)
t = np.arange(n)
demand_C   = np.clip(500 + 300*np.sin(2*np.pi*t/52) + np.random.normal(0,40,n), 0, None)
forecast_C = np.full(n, 500.0)  # naive planner ignores seasonality

products = {
    'Product A (FMCG — stable)':      (demand_A, forecast_A),
    'Product B (Industrial — spiky)': (demand_B, forecast_B),
    'Product C (Apparel — seasonal)': (demand_C, forecast_C),
}

print('Three demand profiles generated')
for name, (d, f) in products.items():
    print(f'  {name}: mean={d.mean():.0f}, std={d.std():.0f}')

In [ ]:
# ── 2. FULL KPI SUITE ─────────────────────────────────────────────────
def full_kpi_suite(actual, forecast):
    """
    Complete KPI suite — Vandeput Chapter 2.
    All formulas implemented from scratch.
    """
    a, f = np.array(actual), np.array(forecast)
    errors = f - a
    dem_avg = a.mean()

    # Bias — systematic direction
    bias_abs = errors.mean()
    bias_pct = bias_abs / dem_avg * 100

    # MAE — average absolute miss
    mae = np.abs(errors).mean()
    mae_pct = mae / dem_avg * 100

    # RMSE — penalises large errors more
    rmse = np.sqrt((errors**2).mean())

    # MAPE — percentage error (Vandeput warns: biased toward under-forecast)
    nonzero = a != 0
    mape = np.mean(np.abs(errors[nonzero] / a[nonzero])) * 100

    # Tracking signal — cumulative bias / MAE ratio
    tracking_signal = np.cumsum(errors) / (np.abs(errors).mean() + 1e-8)

    return {
        'Bias (abs)':    round(bias_abs, 1),
        'Bias (%)':      round(bias_pct, 1),
        'MAE':           round(mae, 1),
        'MAE (%)':       round(mae_pct, 1),
        'RMSE':          round(rmse, 1),
        'RMSE/MAE':      round(rmse / mae, 2),  # >1.3 = outlier-dominated
        'MAPE (%)':      round(mape, 1),
        'Direction':     'Over' if bias_abs > 0 else 'Under',
        '_tracking':     tracking_signal,
        '_errors':       errors,
    }

kpi_results = {name: full_kpi_suite(d, f) for name, (d,f) in products.items()}

# Display table
display_cols = ['Bias (abs)', 'Bias (%)', 'MAE', 'MAE (%)', 'RMSE', 'RMSE/MAE', 'MAPE (%)', 'Direction']
df_kpi = pd.DataFrame({k: {c: v[c] for c in display_cols}
                        for k, v in kpi_results.items()}).T
print('=== KPI COMPARISON — ALL PRODUCTS ===')
print(df_kpi.to_string())
print()
print('NOTE: RMSE/MAE > 1.3 indicates outlier-dominated errors (Product B = 2.8 — spikes matter!)')

In [ ]:
# ── 3. VISUAL KPI DASHBOARD ───────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

prod_names  = list(products.keys())
prod_colors = ['#0077b6', '#e07b00', '#6246ea']

for row, (name, (d, f)) in enumerate(products.items()):
    kpis = kpi_results[name]
    color = prod_colors[row]

    # Demand vs Forecast
    ax1 = fig.add_subplot(gs[row, 0])
    ax1.plot(d, color='#1a2332', lw=1.4, label='Actual')
    ax1.plot(f, color=color, lw=1.4, ls='--', label='Forecast')
    ax1.set_title(name.split('(')[0].strip(), fontsize=9, fontweight='bold')
    ax1.legend(fontsize=7)
    ax1.set_ylabel('Units', fontsize=8)

    # Error distribution
    ax2 = fig.add_subplot(gs[row, 1])
    errors = kpis['_errors']
    ax2.hist(errors, bins=20, color=color, alpha=0.7, edgecolor='white')
    ax2.axvline(0, color='black', lw=1, ls='--')
    ax2.axvline(kpis['Bias (abs)'], color='#c0392b', lw=1.5,
                label=f'Bias={kpis["Bias (abs)"]:.0f}')
    ax2.set_title('Error Distribution', fontsize=9, fontweight='bold')
    ax2.set_xlabel('Forecast Error', fontsize=8)
    ax2.legend(fontsize=7)

    # Tracking signal
    ax3 = fig.add_subplot(gs[row, 2])
    ts = kpis['_tracking']
    ax3.plot(ts, color=color, lw=1.4)
    ax3.axhline(0, color='black', lw=0.8, ls='--')
    ax3.axhline(4,  color='#c0392b', lw=1, ls=':', alpha=0.7)
    ax3.axhline(-4, color='#c0392b', lw=1, ls=':', alpha=0.7)
    ax3.fill_between(range(len(ts)), -4, 4, alpha=0.05, color='green')
    ax3.set_title('Tracking Signal (±4 = out of control)', fontsize=9, fontweight='bold')
    ax3.set_xlabel('Week', fontsize=8)
    ax3.set_ylabel('TS', fontsize=8)

    # Annotate KPI values on demand chart
    ax1.text(0.02, 0.08,
             f'MAE%={kpis["MAE (%)"]:.1f}%  Bias={kpis["Bias (abs)"]:.0f}',
             transform=ax1.transAxes, fontsize=7,
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

fig.suptitle('Forecast KPI Dashboard — 3 Products, Same Metric, Different Stories',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('output_ch02_kpi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch02_kpi_dashboard.png')

In [ ]:
# ── 4. THE MAPE WARNING ───────────────────────────────────────────────
# Demonstrate Vandeput's MAPE criticism:
# MAPE penalises over-forecasting more than under-forecasting
n_test = 50
actual = np.full(n_test, 100.0)

# Equal absolute errors, different directions
over_forecast  = actual + 20   # always 20 units too high
under_forecast = actual - 20   # always 20 units too low

mape_over  = np.mean(np.abs((over_forecast  - actual) / actual)) * 100
mape_under = np.mean(np.abs((under_forecast - actual) / actual)) * 100
mae_over   = np.abs(over_forecast  - actual).mean()
mae_under  = np.abs(under_forecast - actual).mean()

print('=== MAPE ASYMMETRY DEMONSTRATION ===')
print(f'  Same absolute error (20 units), but different direction:')
print(f'  Over-forecast  → MAE={mae_over:.0f} | MAPE={mape_over:.1f}%')
print(f'  Under-forecast → MAE={mae_under:.0f} | MAPE={mape_under:.1f}%')
print(f'  MAE is identical ✓ — MAPE is identical too in this symmetric case.')
print()
print('  But with low-demand periods (d=10):')
actual_low = np.array([10., 100., 10., 100., 10.])
forecast_l = np.array([30.,  80., 30.,  80., 30.])
mape_mixed = np.mean(np.abs((forecast_l - actual_low) / actual_low)) * 100
mae_mixed  = np.abs(forecast_l - actual_low).mean()
print(f'  Mixed demands → MAE={mae_mixed:.0f} units | MAPE={mape_mixed:.0f}%')
print(f'  The low-demand weeks (error=20 on demand=10) dominate MAPE.')
print(f'  MAPE is skewed by small denominators. Use MAE% instead.')

In [ ]:
# ── 5. KPI SELECTION GUIDE ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

kpi_guide = [
    ['KPI', 'Formula', 'Use When', 'Avoid When', 'Supply Chain Application'],
    ['Bias', 'mean(f-d)', 'Detecting systematic error\ndirection',
     'As sole accuracy metric', 'Safety stock tuning\nOver/under-buy detection'],
    ['MAE', 'mean(|f-d|)', 'Primary accuracy metric\nAll demand types',
     'Almost never', 'KPI reporting\nModel comparison\nService level planning'],
    ['RMSE', 'sqrt(mean(e²))', 'When large errors are\ncostly (stockouts)',
     'Noisy data with outliers', 'High-value SKUs\nProduction scheduling'],
    ['MAPE', 'mean(|e/d|)%', 'Cross-product comparison\n(same scale)',
     'Low-volume items (d<10)', 'Portfolio-level reporting\n(with caution)'],
    ['FVA', '(MAE_naive-MAE)/MAE_naive', 'Proving model value\nvs baseline',
     'Standalone (needs baseline)', 'S&OP model selection\nJustify DS investment'],
]

colors = [['#0077b6']*5] + [['#f8f9fb']*5 if i%2==0 else ['white']*5
                               for i in range(len(kpi_guide)-1)]
table = ax.table(cellText=kpi_guide[1:], colLabels=kpi_guide[0],
                 cellLoc='center', loc='center', bbox=[0,0,1,1])
table.auto_set_font_size(False); table.set_fontsize(8)
for (r,c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#0077b6'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f8f9fb')
    cell.set_edgecolor('#e2e6ec')
    cell.set_height(0.18)

ax.set_title('KPI Selection Guide — Vandeput Chapter 2',
             fontsize=12, fontweight='bold', pad=10, y=1.02)
plt.savefig('output_ch02_kpi_guide.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch02_kpi_guide.png')

## Summary

| Product | Main Issue | KPI That Reveals It | Business Action |
|---------|-----------|--------------------|-----------------|
| A (FMCG stable) | Slight systematic under-forecast | Bias = -30 | Adjust safety stock up slightly |
| B (Industrial spiky) | RMSE/MAE = 2.8 → outlier-dominated | RMSE >> MAE | Use RMSE; plan buffer stock for spike weeks |
| C (Apparel seasonal) | Tracking signal exits control bounds | Tracking signal | Model doesn't capture seasonality — upgrade model |

**Key lesson:** The same forecast can look acceptable in MAE but catastrophic in tracking signal.  
Always check Bias AND accuracy — they reveal different failure modes.

**Next:** `03_exponential_smoothing.ipynb` — solves Product A's bias with α-controlled smoothing